# Predicción de la respuesta más preferida

El objetivo es predecir, para una pregunta con varias respuestas candidatas, cuál de ellas recibe la mayor preferencia de los usuarios.

Tras comprobar que el preference_score y los upvotes seleccionan prácticamente siempre la misma respuesta top-1, se utiliza directamente el número de upvotes como criterio objetivo.

Para cada pregunta, las respuestas con el máximo número de upvotes se consideran correctas. Los casos con empate se excluyen del entrenamiento, pero se mantienen en evaluación, donde cualquiera de las respuestas empatadas cuenta como acierto.

Se utiliza un split 80/20 por questionID y validación cruzada de 5 folds sobre train para comparar los distintos enfoques.

## Carga y limpieza de datos

Se carga el dataset original de CounselChat y se realiza una preparación básica para el modelado. Se consideran únicamente preguntas con al menos dos respuestas y con alguna señal de preferencia, es decir, aquellas en las que al menos una respuesta ha recibido algún upvote.

In [1]:
import numpy as np
import pandas as pd

In [2]:
url = (
    "https://huggingface.co/datasets/nbertagnolli/counsel-chat/"
    "raw/main/20220401_counsel_chat.csv"
)

df = pd.read_csv(url)

print("Filas:", len(df))
print("Preguntas:", df["questionID"].nunique())

Filas: 2775
Preguntas: 940


Vuelvo a realizar comprobaciones básicas:

In [3]:
columnas_modelo = [
    "questionID",
    "questionText",
    "answerText",
    "upvotes"
]

print("Valores nulos:")
display(df[columnas_modelo].isna().sum())

print("Tipo de upvotes:", df["upvotes"].dtype)

Valores nulos:


,0
questionID,0
questionText,139
answerText,26
upvotes,0


Tipo de upvotes: int64


Como no se observan valores ausentes en las variables utilizadas y upvotes ya presenta formato numérico, no es necesaria una limpieza adicional de estas columnas.

In [4]:
df_modelo = df.copy()

n_respuestas = (
    df_modelo.groupby("questionID")["answerText"]
    .transform("size")
)

df_modelo = df_modelo[
    n_respuestas >= 2
].copy()

print(
    "Preguntas con al menos 2 respuestas:",
    df_modelo["questionID"].nunique()
)

print(
    "Respuestas incluidas:",
    len(df_modelo)
)

Preguntas con al menos 2 respuestas: 460
Respuestas incluidas: 2295


Dado que el objetivo es seleccionar la mejor respuesta entre varias candidatas, se excluyen las preguntas con una única respuesta.

In [5]:
max_upvotes = (
    df_modelo.groupby("questionID")["upvotes"]
    .transform("max")
)

df_modelo = df_modelo[
    max_upvotes > 0
].copy()

print(
    "Preguntas con señal de preferencia:",
    df_modelo["questionID"].nunique()
)

print(
    "Respuestas finales:",
    len(df_modelo)
)

Preguntas con señal de preferencia: 276
Respuestas finales: 1732


También se excluyen las preguntas en las que todas las respuestas tienen 0 upvotes, ya que en esos casos no existe una señal observable que permita identificar una respuesta preferida.

Comprobación visual de cómo ha quedado df_modelo después de los filtros:

In [6]:
df_modelo[
    [
        "questionID",
        "questionText",
        "answerText",
        "upvotes"
    ]
].head()

,questionID,questionText,answerText,upvotes
0,0,I have so many issues to address. I have a his...,It is very common for people to have multiple ...,3
1,0,I have so many issues to address. I have a his...,"I've never heard of someone having ""too many i...",2
2,0,I have so many issues to address. I have a his...,Absolutely not. I strongly recommending worki...,2
3,0,I have so many issues to address. I have a his...,Let me start by saying there are never too man...,2
4,0,I have so many issues to address. I have a his...,I just want to acknowledge you for the courage...,1


## Definición de la variable objetivo y análisis de empates

Para cada pregunta, se consideran como respuestas preferidas aquellas que tienen el número máximo de upvotes. Estas respuestas reciben target = 1, mientras que el resto recibe target = 0.

Como pueden existir varias respuestas con el mismo número máximo de upvotes, se identifican también las preguntas con empate. Estos casos se mantendrán por ahora en el dataset para poder tratarlos posteriormente de forma diferente en entrenamiento y evaluación.

Creo el target:

In [7]:
df_modelo["target"] = (
    df_modelo["upvotes"]
    == df_modelo.groupby("questionID")["upvotes"].transform("max")
).astype(int)

Apunte: Si dos respuestas están empatadas en el máximo, las dos tendrán target = 1.

Identifico empates:

In [8]:
n_top_por_pregunta = (
    df_modelo.groupby("questionID")["target"]
    .sum()
)

preguntas_empate = n_top_por_pregunta[
    n_top_por_pregunta > 1
].index

preguntas_unica_top = n_top_por_pregunta[
    n_top_por_pregunta == 1
].index

print("Preguntas:", len(n_top_por_pregunta))
print(
    "Con una única respuesta top:",
    len(preguntas_unica_top)
)
print(
    "Con empate en máximo upvotes:",
    len(preguntas_empate)
)
print(
    "Porcentaje con empate:",
    f"{len(preguntas_empate) / len(n_top_por_pregunta) * 100:.2f}%"
)

Preguntas: 276
Con una única respuesta top: 214
Con empate en máximo upvotes: 62
Porcentaje con empate: 22.46%


Se observan 62 preguntas con empate en el máximo número de upvotes (22,46 %), mientras que 214 preguntas presentan una única respuesta top.

Los casos con empate se mantienen por ahora en el dataset. Tras realizar el split, se excluirán del entrenamiento y se conservarán en evaluación, donde se considerará correcta cualquiera de las respuestas empatadas en el máximo.

## División train-test

Dado el tamaño reducido del dataset, se utiliza una división del 80 % para entrenamiento y 20 % para test.

La separación se realiza por questionID, de forma que todas las respuestas de una misma pregunta permanezcan siempre en el mismo conjunto.

Primero se realiza el split sobre las 276 preguntas disponibles. Posteriormente, las preguntas con empate en el máximo número de upvotes se eliminan únicamente del conjunto de entrenamiento. El conjunto de test se mantiene completo, incluyendo los casos con empate.

Realizo el spit:

In [9]:
from sklearn.model_selection import train_test_split

question_ids = df_modelo["questionID"].unique()

train_ids, test_ids = train_test_split(
    question_ids,
    test_size=0.20,
    random_state=27
)

train_completo = df_modelo[
    df_modelo["questionID"].isin(train_ids)
].copy()

test = df_modelo[
    df_modelo["questionID"].isin(test_ids)
].copy()

print(
    "Preguntas en train antes de eliminar empates:",
    train_completo["questionID"].nunique()
)

print(
    "Preguntas en test:",
    test["questionID"].nunique()
)

Preguntas en train antes de eliminar empates: 220
Preguntas en test: 56


Las preguntas con empate se eliminan únicamente de train, ya que durante el entrenamiento se quiere disponer de una única respuesta objetivo por pregunta. En cambio, permanecen en test para poder evaluar el modelo también sobre estos casos.

Elimino empates solo del train:

In [10]:
train = train_completo[
    ~train_completo["questionID"].isin(preguntas_empate)
].copy()

print(
    "Preguntas en train después de eliminar empates:",
    train["questionID"].nunique()
)

print(
    "Preguntas eliminadas de train por empate:",
    train_completo["questionID"].nunique()
    - train["questionID"].nunique()
)

print(
    "Preguntas en test:",
    test["questionID"].nunique()
)

Preguntas en train después de eliminar empates: 172
Preguntas eliminadas de train por empate: 48
Preguntas en test: 56


Antes de continuar compruebo que el split es correcto:

In [11]:
# Compruebo que no haya preguntas compartidas entre train y test
preguntas_comunes = (
    set(train["questionID"])
    & set(test["questionID"])
)

# Compruebo que en train no haya quedado ninguna pregunta con varias respuestas target = 1
targets_train = (
    train.groupby("questionID")["target"]
    .sum()
)

# En test sí que pueden existir empates, miro cuantos hay
targets_test = (
    test.groupby("questionID")["target"]
    .sum()
)

print(
    "Preguntas compartidas entre train y test:",
    len(preguntas_comunes)
)

print(
    "Preguntas con empate que quedan en train:",
    (targets_train > 1).sum()
)

print(
    "Preguntas con empate presentes en test:",
    (targets_test > 1).sum()
)

Preguntas compartidas entre train y test: 0
Preguntas con empate que quedan en train: 0
Preguntas con empate presentes en test: 14


De esta forma, el conjunto de entrenamiento contiene únicamente preguntas con una respuesta top inequívoca, mientras que el conjunto de test conserva tanto preguntas con una única respuesta preferida como preguntas con varias respuestas empatadas en el máximo de upvotes.

In [12]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda x: f"{x:.6f}")

In [13]:
# Fuente pública de CounselChat con las columnas upvotes y views
DATA_URL = (
    "https://huggingface.co/datasets/nbertagnolli/counsel-chat/"
    "raw/main/20220401_counsel_chat.csv"
)

# Si no hay conexión, el cuaderno intentará usar alguno de estos archivos
# si está en la misma carpeta que el notebook
LOCAL_CANDIDATES = [
    Path("20220401_counsel_chat.csv"),
    Path("counselchat-data.csv"),
]

try:
    df = pd.read_csv(DATA_URL)
    fuente = DATA_URL
except Exception:
    archivo_local = next((p for p in LOCAL_CANDIDATES if p.exists()), None)
    if archivo_local is None:
        raise FileNotFoundError(
            "No se pudo descargar el dataset y tampoco se encontró un CSV local. "
            "Coloca '20220401_counsel_chat.csv' en la misma carpeta del notebook."
        )
    df = pd.read_csv(archivo_local)
    fuente = str(archivo_local)

print(f"Fuente: {fuente}")
print(f"Filas: {len(df):,}")
print(f"Preguntas únicas: {df['questionID'].nunique():,}")
print("\nColumnas:")
print(df.columns.tolist())

Fuente: https://huggingface.co/datasets/nbertagnolli/counsel-chat/raw/main/20220401_counsel_chat.csv
Filas: 2,775
Preguntas únicas: 940

Columnas:
['questionID', 'questionTitle', 'questionText', 'questionLink', 'topic', 'therapistInfo', 'therapistURL', 'answerText', 'upvotes', 'views']


Para que exista un problema de selección entre respuestas, se consideran únicamente preguntas con al menos dos respuestas. Además, se eliminan las preguntas en las que ninguna respuesta ha recibido upvotes, ya que en esos casos no existe una señal observable de preferencia.

In [14]:
df_modelo = df.copy()

# Aseguramos que upvotes sea numérico
df_modelo["upvotes"] = pd.to_numeric(
    df_modelo["upvotes"],
    errors="coerce"
)

# Eliminamos filas sin información esencial
df_modelo = df_modelo.dropna(
    subset=[
        "questionID",
        "questionText",
        "answerText",
        "upvotes"
    ]
).copy()

# Preguntas con al menos 2 respuestas
n_respuestas = df_modelo.groupby("questionID").size()

ids_multi = n_respuestas[
    n_respuestas >= 2
].index

df_modelo = df_modelo[
    df_modelo["questionID"].isin(ids_multi)
].copy()

# Preguntas con al menos un upvote positivo
max_upvotes = (
    df_modelo.groupby("questionID")["upvotes"]
    .max()
)

ids_con_senal = max_upvotes[
    max_upvotes > 0
].index

df_modelo = df_modelo[
    df_modelo["questionID"].isin(ids_con_senal)
].copy()

print(
    "Preguntas con al menos 2 respuestas:",
    len(ids_multi)
)

print(
    "Preguntas con señal de preferencia:",
    df_modelo["questionID"].nunique()
)

print(
    "Respuestas incluidas:",
    len(df_modelo)
)

Preguntas con al menos 2 respuestas: 427
Preguntas con señal de preferencia: 262
Respuestas incluidas: 1667


Para cada pregunta, se consideran respuestas preferidas aquellas que presentan el número máximo de upvotes. Estas reciben target = 1, mientras que el resto recibe target = 0.

In [15]:
df_modelo["target"] = (
    df_modelo["upvotes"]
    == df_modelo.groupby("questionID")["upvotes"].transform("max")
).astype(int)

In [16]:
ganadoras = (
    df_modelo.groupby("questionID")["target"]
    .sum()
)

print("Preguntas:", len(ganadoras))

print(
    "Con una única respuesta top:",
    (ganadoras == 1).sum()
)

print(
    "Con empate en máximo upvotes:",
    (ganadoras > 1).sum()
)

print(
    "Porcentaje con empate:",
    f"{(ganadoras > 1).mean() * 100:.2f}%"
)

Preguntas: 262
Con una única respuesta top: 202
Con empate en máximo upvotes: 60
Porcentaje con empate: 22.90%


## Validación cruzada sobre train

Como no se reserva un conjunto de validación independiente, se utiliza una validación cruzada de 5 folds sobre el conjunto de entrenamiento.

En cada iteración, cuatro folds se utilizan para entrenamiento y el fold restante para validación. El proceso se repite cinco veces, de forma que todas las preguntas de train se utilizan una vez como validación.

La división se realiza agrupando por questionID, para garantizar que todas las respuestas de una misma pregunta permanezcan siempre en el mismo fold.

Creo los folds:

In [17]:
from sklearn.model_selection import GroupKFold

cv = GroupKFold(n_splits=5)

cv_splits = list(
    cv.split(
        train,
        groups=train["questionID"]
    )
)

print("Número de folds:", len(cv_splits))

Número de folds: 5


Compruebo cómo queda cada fold:

In [18]:
for fold, (train_idx, val_idx) in enumerate(
    cv_splits,
    start=1
):
    train_fold = train.iloc[train_idx]
    val_fold = train.iloc[val_idx]

    preguntas_train_fold = set(
        train_fold["questionID"]
    )

    preguntas_val_fold = set(
        val_fold["questionID"]
    )

    preguntas_comunes = (
        preguntas_train_fold
        & preguntas_val_fold
    )

    print(
        f"Fold {fold}: "
        f"{len(preguntas_train_fold)} preguntas train | "
        f"{len(preguntas_val_fold)} preguntas validación | "
        f"{len(preguntas_comunes)} compartidas"
    )

Fold 1: 141 preguntas train | 31 preguntas validación | 0 compartidas
Fold 2: 139 preguntas train | 33 preguntas validación | 0 compartidas
Fold 3: 137 preguntas train | 35 preguntas validación | 0 compartidas
Fold 4: 135 preguntas train | 37 preguntas validación | 0 compartidas
Fold 5: 136 preguntas train | 36 preguntas validación | 0 compartidas


En todos los folds aparecen 0 compartidas, eso nos confirma que una misma pregunta nunca aparece a la vez en entrenamiento y validación.

In [19]:
preguntas_validadas = set()

for _, val_idx in cv_splits:
    preguntas_validadas.update(
        train.iloc[val_idx]["questionID"].unique()
    )

print(
    "Preguntas totales en train:",
    train["questionID"].nunique()
)

print(
    "Preguntas utilizadas alguna vez como validación:",
    len(preguntas_validadas)
)

Preguntas totales en train: 172
Preguntas utilizadas alguna vez como validación: 172


Los 5 folds mantienen agrupadas las respuestas por questionID y no existe solapamiento de preguntas entre entrenamiento y validación dentro de cada iteración. Además, todas las preguntas de train se utilizan una vez como conjunto de validación.

## Baselines simples

Antes de evaluar modelos entrenados, se utilizan varios baselines sencillos como referencia:

- Selección aleatoria de una respuesta.
- Selección de la respuesta más larga.
- Selección de la respuesta más corta.
- Selección de la respuesta con mayor solapamiento léxico con la pregunta.

Los baselines se evalúan sobre el mismo conjunto de test mediante Accuracy@1. En las preguntas con empate en el máximo número de upvotes, se considera correcta cualquiera de las respuestas con target = 1.

Estos métodos no requieren entrenamiento, por lo que no utilizan la validación cruzada definida sobre train.

### Baseline aleatorio

El baseline aleatorio representa el rendimiento esperado al seleccionar uniformemente una de las respuestas candidatas de cada pregunta. Si existen varias respuestas empatadas como correctas, cualquiera de ellas constituye un acierto. En lugar de realizar una única selección aleatoria, se calcula directamente la probabilidad esperada de acierto para obtener un resultado estable.

In [ ]:
random_por_pregunta = (
    test.groupby("questionID")
    .agg(
        n_respuestas=("target", "size"),
        n_correctas=("target", "sum")
    )
)

accuracy_random = (
    random_por_pregunta["n_correctas"]
    / random_por_pregunta["n_respuestas"]
).mean()

print(
    f"Baseline aleatorio, Accuracy@1: "
    f"{accuracy_random:.2%}"
)

Baseline aleatorio, Accuracy@1: 39.97%


### Baseline respuesta más larga y más corta

Este baseline comprueba si una regla muy simple basada únicamente en la longitud de la respuesta puede identificar la respuesta preferida. Se evalúan dos variantes: seleccionar la respuesta más larga y seleccionar la más corta. La longitud se mide mediante el número de palabras de cada respuesta.

In [ ]:
test["longitud_respuesta"] = (
    test["answerText"]
    .fillna("")
    .str.split()
    .str.len()
)

Si varias respuestas empatan en la longitud máxima o mínima, se selecciona una de ellas al azar de forma reproducible, evitando favorecer siempre a la respuesta que aparece primero en el dataset.

In [ ]:
def seleccionar_por_metrica(datos, columna, modo="max", seed=27):
    rng = np.random.default_rng(seed)
    indices_seleccionados = []

    for _, grupo in datos.groupby("questionID"):

        if modo == "max":
            mejor_valor = grupo[columna].max()
        else:
            mejor_valor = grupo[columna].min()

        candidatas = grupo.index[
            grupo[columna] == mejor_valor
        ].to_numpy()

        indices_seleccionados.append(
            rng.choice(candidatas)
        )

    return datos.loc[indices_seleccionados]

In [ ]:
pred_larga = seleccionar_por_metrica(
    test,
    "longitud_respuesta",
    modo="max"
)

accuracy_larga = pred_larga["target"].mean()

print(
    f"Respuesta más larga, Accuracy@1: "
    f"{accuracy_larga:.2%}"
)

Respuesta más larga, Accuracy@1: 51.79%


In [ ]:
pred_corta = seleccionar_por_metrica(
    test,
    "longitud_respuesta",
    modo="min"
)

accuracy_corta = pred_corta["target"].mean()

print(
    f"Respuesta más corta, Accuracy@1: "
    f"{accuracy_corta:.2%}"
)

Respuesta más corta, Accuracy@1: 32.14%


Como apunte, se recuerda que el target puede ser 1 para varias respuestas empatadas por upvotes, si el baseline elige cualquiera de ellas cuenta correctamente como acierto.

### Baseline Solapamiento léxico

Como baseline textual sencillo, se selecciona la respuesta que comparte una mayor proporción de palabras relevantes con la pregunta. Se eliminan las stopwords en inglés antes de calcular el solapamiento.

In [ ]:
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

stopwords = set(ENGLISH_STOP_WORDS)


def obtener_palabras(texto):
    palabras = re.findall(
        r"[a-zA-Z']+",
        str(texto).lower()
    )

    return {
        palabra
        for palabra in palabras
        if palabra not in stopwords
    }


def calcular_solapamiento(fila):
    palabras_pregunta = obtener_palabras(
        fila["questionText"]
    )

    palabras_respuesta = obtener_palabras(
        fila["answerText"]
    )

    if len(palabras_pregunta) == 0:
        return 0

    return (
        len(palabras_pregunta & palabras_respuesta)
        / len(palabras_pregunta)
    )

In [ ]:
test["solapamiento_lexico"] = test.apply(
    calcular_solapamiento,
    axis=1
)

pred_solapamiento = seleccionar_por_metrica(
    test,
    "solapamiento_lexico",
    modo="max"
)

accuracy_solapamiento = (
    pred_solapamiento["target"].mean()
)

print(
    f"Solapamiento léxico, Accuracy@1: "
    f"{accuracy_solapamiento:.2%}"
)

Solapamiento léxico, Accuracy@1: 41.07%


### Resumen de baselines

In [ ]:
resultados_baselines = pd.DataFrame({
    "Método": [
        "Aleatorio",
        "Respuesta más larga",
        "Respuesta más corta",
        "Solapamiento léxico"
    ],
    "Accuracy@1 (%)": [
        accuracy_random * 100,
        accuracy_larga * 100,
        accuracy_corta * 100,
        accuracy_solapamiento * 100
    ]
})

display(
    resultados_baselines.round(2)
)

,Método,Accuracy@1 (%)
0,Aleatorio,39.970000
1,Respuesta más larga,51.790000
2,Respuesta más corta,32.140000
3,Solapamiento léxico,41.070000


Estos resultados establecen referencias sencillas con las que comparar posteriormente los modelos entrenados. En particular, permiten comprobar si un modelo aporta información adicional frente a señales básicas como la longitud de la respuesta o el solapamiento de palabras con la pregunta.

## TF-IDF + Regresión Logística

Como primer modelo entrenado, se utiliza TF-IDF + Regresión Logística.


El proceso es el siguiente:

- Para cada respuesta candidata se combina el texto de la pregunta + texto de la respuesta.
- TF-IDF transforma cada combinación de texto en una representación numérica, teniendo en cuenta las palabras que aparecen y su importancia.
- Durante el entrenamiento, el modelo recibe:
  - El texto transformado por TF-IDF.
  - El target, que indica si esa respuesta fue la preferida (1) o no (0).
- El target se obtiene a partir de los upvotes: la respuesta con mayor número de upvotes recibe target = 1 y el resto target = 0.
- La Regresión Logística aprende qué patrones del texto suelen estar asociados con respuestas preferidas.
- En validación y test, el modelo ya no recibe ni los upvotes ni el target. Solo recibe el texto y estima una probabilidad de que cada respuesta sea la preferida.
- Finalmente, para cada pregunta se selecciona la respuesta con la probabilidad más alta.

Preparación del texto:

In [ ]:
train["texto"] = (
    train["questionText"].fillna("")
    + " "
    + train["answerText"].fillna("")
)

test["texto"] = (
    test["questionText"].fillna("")
    + " "
    + test["answerText"].fillna("")
)

#### Validación cruzada sobre train

Antes de evaluar el modelo sobre test, se realiza una validación cruzada de 5 folds sobre train.

En cada iteración, 4 folds se utilizan para entrenar y 1 fold se utiliza para validar.

TF-IDF aprende únicamente con los textos de los 4 folds de entrenamiento. Después, utiliza lo aprendido para convertir en números los textos del fold de validación, sin utilizar sus resultados reales para entrenar.

El modelo predice una probabilidad para cada respuesta del fold de validación y, para cada pregunta, se selecciona la respuesta con mayor probabilidad.

El Accuracy@1 se calcula solo sobre ese fold de validación. Este proceso se repite 5 veces y finalmente se calcula la media de los 5 Accuracy@1.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

accuracies_cv = []

for fold, (train_idx, val_idx) in enumerate(
    cv_splits,
    start=1
):
    train_fold = train.iloc[train_idx].copy()
    val_fold = train.iloc[val_idx].copy()

    # TF-IDF se ajusta únicamente con el train del fold
    vectorizer_fold = TfidfVectorizer(
        max_features=5000,
        stop_words="english"
    )

    X_train_fold = vectorizer_fold.fit_transform(
        train_fold["texto"]
    )

    X_val_fold = vectorizer_fold.transform(
        val_fold["texto"]
    )

    # Entrenamiento del modelo
    modelo_fold = LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=27
    )

    modelo_fold.fit(
        X_train_fold,
        train_fold["target"]
    )

    # Probabilidad de que cada respuesta sea la preferida
    val_fold["prob_preferida"] = (
        modelo_fold.predict_proba(X_val_fold)[:, 1]
    )

    # Seleccionamos una respuesta por pregunta:
    # la que tenga mayor probabilidad
    predicciones_fold = val_fold.loc[
        val_fold.groupby("questionID")[
            "prob_preferida"
        ].idxmax()
    ]

    accuracy_fold = (
        predicciones_fold["target"].mean()
    )

    accuracies_cv.append(accuracy_fold)

    print(
        f"Fold {fold} - Accuracy@1: "
        f"{accuracy_fold:.2%}"
    )

print()
print(
    f"Accuracy@1 media CV: "
    f"{np.mean(accuracies_cv):.2%}"
)

print(
    f"Desviación estándar: "
    f"{np.std(accuracies_cv):.2%}"
)

Fold 1 - Accuracy@1: 25.81%
Fold 2 - Accuracy@1: 39.39%
Fold 3 - Accuracy@1: 20.00%
Fold 4 - Accuracy@1: 29.73%
Fold 5 - Accuracy@1: 27.78%

Accuracy@1 media CV: 28.54%
Desviación estándar: 6.33%


La validación cruzada obtiene una Accuracy@1 media de 27,83 %, con una desviación estándar de 6,05 %. Los resultados varían entre 20,00 % y 36,36 % según el fold, lo que indica que el rendimiento del modelo depende en cierta medida de las preguntas utilizadas para entrenamiento y validación.

En conjunto, TF-IDF + Regresión Logística muestra un rendimiento limitado y cierta variabilidad entre particiones, por lo que será útil compararlo posteriormente con modelos capaces de representar mejor el significado semántico del texto.

#### Entrenamiento final

Tras la validación cruzada, el modelo se entrena de nuevo utilizando todo el conjunto de train, ya sin preguntas empatadas.

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=5000,
    stop_words="english"
)

X_train = vectorizer.fit_transform(
    train["texto"]
)

X_test = vectorizer.transform(
    test["texto"]
)

modelo = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=27
)

modelo.fit(
    X_train,
    train["target"]
)

LogisticRegression(class_weight='balanced', max_iter=1000, random_state=27)

#### Evaluación final en test

El modelo asigna una probabilidad a cada respuesta candidata y, para cada pregunta de test, se selecciona la respuesta con mayor probabilidad.

La predicción se considera correcta si la respuesta seleccionada tiene target = 1. En las preguntas con empate en el máximo número de upvotes, cualquiera de las respuestas empatadas se considera correcta.

In [ ]:
test["prob_preferida"] = (
    modelo.predict_proba(X_test)[:, 1]
)

predicciones = test.loc[
    test.groupby("questionID")[
        "prob_preferida"
    ].idxmax()
]

accuracy_top1 = (
    predicciones["target"].mean()
)

print(
    f"TF-IDF + Regresión Logística "
    f", Accuracy@1 test: "
    f"{accuracy_top1:.2%}"
)

TF-IDF + Regresión Logística , Accuracy@1 test: 33.93%


#### Comparación con los baselines

In [ ]:
resultados = pd.DataFrame({
    "Método": [
        "Aleatorio",
        "Respuesta más larga",
        "Respuesta más corta",
        "Solapamiento léxico",
        "TF-IDF + Regresión Logística"
    ],
    "Accuracy@1 (%)": [
        accuracy_random * 100,
        accuracy_larga * 100,
        accuracy_corta * 100,
        accuracy_solapamiento * 100,
        accuracy_top1 * 100
    ]
})

display(
    resultados.round(2)
)

,Método,Accuracy@1 (%)
0,Aleatorio,39.970000
1,Respuesta más larga,51.790000
2,Respuesta más corta,32.140000
3,Solapamiento léxico,41.070000
4,TF-IDF + Regresión Logística,33.930000


El baseline con mejor rendimiento es seleccionar la respuesta más larga, con una Accuracy@1 del 51,79 %. Esto sugiere que la longitud de la respuesta puede estar asociada con la preferencia de los usuarios.

El solapamiento léxico obtiene un 41,07 %, muy próximo al baseline aleatorio (39,97 %), mientras que la respuesta más corta alcanza un 32,14 %.

TF-IDF + Regresión Logística obtiene un 33,93 %, por debajo del baseline aleatorio y notablemente por debajo de la respuesta más larga. Por tanto, este modelo sencillo basado en palabras no supera los baselines más simples.

Apunte: que el baseline aleatorio sea ahora 39,97 % tiene sentido porque en test conservamos los empates. Si una pregunta tiene, por ejemplo, 2 respuestas correctas de 4, el azar tiene un 50 % de probabilidad de acertar, no un 25 %.

## Modelo de preferencias por pares con BERT

Como enfoque más avanzado, se entrena un modelo BERT para aprender preferencias entre pares de respuestas.

Una pregunta puede tener dos o más respuestas candidatas. En lugar de evaluar cada respuesta por separado, se generan todas las comparaciones posibles entre pares de respuestas de una misma pregunta.

Por ejemplo, si una pregunta tiene tres respuestas A, B y C, se generan:

- A vs B
- A vs C
- B vs C

Durante el entrenamiento, BERT recibe:

- El texto de la pregunta.
- Los textos de las dos respuestas que se están comparando.
- Una etiqueta que indica cuál de las dos respuestas es la preferida.

Esta etiqueta se obtiene utilizando los upvotes: la respuesta con más upvotes se considera la preferida. Los valores de upvotes no se introducen como entrada del modelo, sino que únicamente sirven para crear la solución correcta con la que BERT aprende.

Aunque previamente se eliminaron de train las preguntas con empate en la respuesta top, todavía pueden existir empates entre respuestas no ganadoras. Por ejemplo: A = 5 upvotes, B = 2 y C = 2. La pregunta puede utilizarse porque A es claramente la respuesta top, pero el par B vs C se descarta porque ambas respuestas tienen los mismos upvotes y no se puede determinar cuál es preferida.

Además, cada comparación se introduce también en orden inverso. Por ejemplo, si B es preferida frente a A:

- A vs B: gana B
- B vs A: sigue ganando B

Esto ayuda a evitar que el modelo aprenda un sesgo relacionado con la posición de las respuestas.

En validación y test, BERT ya no recibe los upvotes ni la solución correcta. Únicamente recibe la pregunta y cada par de respuestas, y predice cuál de las dos considera preferida.

Finalmente, si una pregunta tiene varias respuestas, se comparan todas dos a dos y cada respuesta acumula una victoria cuando es seleccionada frente a otra. La respuesta con mayor número de victorias se selecciona como respuesta top-1.

Instalación e importaciones:

In [20]:
!pip -q install transformers datasets accelerate

In [21]:
import gc
import torch

from itertools import combinations
from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    set_seed
)

set_seed(27)

#### Construcción de los pares de entrenamiento

A partir de las respuestas de cada pregunta se generan todas las comparaciones posibles dos a dos.

La etiqueta indica qué posición contiene la respuesta preferida:

- label = 0: la primera respuesta es preferida;
- label = 1: la segunda respuesta es preferida.

Los pares con el mismo número de upvotes se excluyen porque no existe un ganador claro.

In [22]:
def crear_pares_entrenamiento(datos):
    ejemplos = []

    for question_id, grupo in datos.groupby("questionID"):

        filas = list(grupo.itertuples(index=False))

        for respuesta_1, respuesta_2 in combinations(filas, 2):

            # Si tienen los mismos upvotes,
            # no existe una preferencia clara
            if respuesta_1.upvotes == respuesta_2.upvotes:
                continue

            pregunta = str(respuesta_1.questionText)

            texto_1 = str(respuesta_1.answerText)
            texto_2 = str(respuesta_2.answerText)

            # 0 = primera respuesta preferida
            # 1 = segunda respuesta preferida
            if respuesta_1.upvotes > respuesta_2.upvotes:
                label = 0
            else:
                label = 1

            # Orden original
            ejemplos.append({
                "questionID": question_id,
                "texto_a": (
                    f"Question: {pregunta}\n"
                    f"Response A: {texto_1}"
                ),
                "texto_b": (
                    f"Response B: {texto_2}"
                ),
                "label": label
            })

            # Mismo par en orden inverso
            ejemplos.append({
                "questionID": question_id,
                "texto_a": (
                    f"Question: {pregunta}\n"
                    f"Response A: {texto_2}"
                ),
                "texto_b": (
                    f"Response B: {texto_1}"
                ),
                "label": 1 - label
            })

    return pd.DataFrame(ejemplos)

Compruebo cuantos pares se generan:

In [23]:
pares_train = crear_pares_entrenamiento(train)

print("Pares de entrenamiento:", len(pares_train))

print("\nDistribución de etiquetas:")
display(
    pares_train["label"]
    .value_counts()
    .sort_index()
)

Pares de entrenamiento: 6748

Distribución de etiquetas:


,count
label,
0,3374
1,3374


Como cada comparación se introduce también al revés, las clases quedan prácticamente equilibradas.

#### Preparación de BERT

Se utiliza bert-base-uncased, ya que CounselChat está formado por textos en inglés.

BERT recibe la pregunta junto con las dos respuestas y aprende a clasificar cuál de las dos es preferida.

In [24]:
MODEL_NAME = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Creamos una función para transformar los textos al formato que necesita BERT:

In [25]:
def preparar_dataset_bert(pares, entrenamiento=True):

    columnas = ["texto_a", "texto_b"]

    if entrenamiento:
        columnas.append("label")

    dataset = Dataset.from_pandas(
        pares[columnas],
        preserve_index=False
    )

    def tokenizar(batch):
        return tokenizer(
            batch["texto_a"],
            batch["texto_b"],
            truncation=True,
            max_length=512
        )

    dataset = dataset.map(
        tokenizar,
        batched=True
    )

    dataset = dataset.remove_columns(
        ["texto_a", "texto_b"]
    )

    if entrenamiento:
        dataset = dataset.rename_column(
            "label",
            "labels"
        )

    return dataset

#### Selección de la respuesta mediante comparaciones dos a dos

El modelo solo sabe comparar dos respuestas. Para obtener una única respuesta top-1 entre todas las candidatas de una pregunta, se comparan todas las respuestas dos a dos.

Cada comparación se realiza en los dos órdenes posibles y se promedian las probabilidades para reducir el posible efecto de la posición.

Cada respuesta acumula una victoria cuando el modelo la considera preferida frente a otra. Finalmente, se selecciona la respuesta con mayor número de victorias.

Si dos respuestas obtienen el mismo número de victorias, se utiliza como desempate la suma de sus probabilidades de preferencia.

In [26]:
def crear_pares_prediccion(datos):

    datos_eval = datos.reset_index(drop=True).copy()

    datos_eval["row_id"] = np.arange(
        len(datos_eval)
    )

    ejemplos = []
    pair_id = 0

    for question_id, grupo in datos_eval.groupby(
        "questionID"
    ):

        filas = list(
            grupo.itertuples(index=False)
        )

        for respuesta_1, respuesta_2 in combinations(
            filas, 2
        ):

            pregunta = str(
                respuesta_1.questionText
            )

            texto_1 = str(
                respuesta_1.answerText
            )

            texto_2 = str(
                respuesta_2.answerText
            )

            # Orden A-B
            ejemplos.append({
                "pair_id": pair_id,
                "questionID": question_id,
                "row_i": respuesta_1.row_id,
                "row_j": respuesta_2.row_id,
                "orden": "ij",
                "texto_a": (
                    f"Question: {pregunta}\n"
                    f"Response A: {texto_1}"
                ),
                "texto_b": (
                    f"Response B: {texto_2}"
                )
            })

            # Orden B-A
            ejemplos.append({
                "pair_id": pair_id,
                "questionID": question_id,
                "row_i": respuesta_1.row_id,
                "row_j": respuesta_2.row_id,
                "orden": "ji",
                "texto_a": (
                    f"Question: {pregunta}\n"
                    f"Response A: {texto_2}"
                ),
                "texto_b": (
                    f"Response B: {texto_1}"
                )
            })

            pair_id += 1

    return (
        datos_eval,
        pd.DataFrame(ejemplos)
    )

Ahora la función que realiza el torneo:

In [27]:
def evaluar_torneo(trainer, datos):

    datos_eval, pares = (
        crear_pares_prediccion(datos)
    )

    dataset = preparar_dataset_bert(
        pares,
        entrenamiento=False
    )

    predicciones = trainer.predict(
        dataset
    )

    logits = torch.tensor(
        predicciones.predictions
    )

    probabilidades = torch.softmax(
        logits,
        dim=1
    ).numpy()

    pares["prob_primera"] = (
        probabilidades[:, 0]
    )

    pares["prob_segunda"] = (
        probabilidades[:, 1]
    )

    victorias = {
        row_id: 0.0
        for row_id in datos_eval["row_id"]
    }

    puntuaciones = {
        row_id: 0.0
        for row_id in datos_eval["row_id"]
    }

    for pair_id, grupo in pares.groupby(
        "pair_id"
    ):

        orden_ij = grupo[
            grupo["orden"] == "ij"
        ].iloc[0]

        orden_ji = grupo[
            grupo["orden"] == "ji"
        ].iloc[0]

        row_i = orden_ij["row_i"]
        row_j = orden_ij["row_j"]

        # Probabilidad de que i sea preferida:
        # cuando aparece primera y cuando aparece segunda
        prob_i = (
            orden_ij["prob_primera"]
            + orden_ji["prob_segunda"]
        ) / 2

        prob_j = 1 - prob_i

        puntuaciones[row_i] += prob_i
        puntuaciones[row_j] += prob_j

        if prob_i > prob_j:
            victorias[row_i] += 1

        elif prob_j > prob_i:
            victorias[row_j] += 1

        else:
            victorias[row_i] += 0.5
            victorias[row_j] += 0.5

    ganadores = []

    for question_id, grupo in datos_eval.groupby(
        "questionID"
    ):

        grupo = grupo.copy()

        grupo["victorias"] = grupo[
            "row_id"
        ].map(victorias)

        grupo["puntuacion"] = grupo[
            "row_id"
        ].map(puntuaciones)

        ganador = grupo.sort_values(
            ["victorias", "puntuacion"],
            ascending=[False, False]
        ).iloc[0]

        ganadores.append(ganador)

    ganadores = pd.DataFrame(
        ganadores
    )

    accuracy = (
        ganadores["target"].mean()
    )

    return accuracy, ganadores

En esta sección se han definido las funciones que se utilizarán durante la evaluación, una vez que BERT ya ha sido entrenado.


#### Validación cruzada de 5 folds

Al igual que con TF-IDF + Regresión Logística, el modelo se evalúa primero mediante los 5 folds definidos sobre train.

En cada iteración:

Cuatro folds se utilizan para crear los pares y entrenar BERT.
El fold restante se utiliza únicamente para validación.
El modelo compara las respuestas del fold de validación dos a dos.
Se selecciona una respuesta top-1 por pregunta.
Accuracy@1 se calcula únicamente sobre las preguntas del fold de validación.

Se obtiene así un Accuracy@1 para cada fold y posteriormente se calcula la media.

In [ ]:
accuracies_bert_cv = []

for fold, (train_idx, val_idx) in enumerate(
    cv_splits,
    start=1
):

    print(f"\n===== FOLD {fold} =====")

    train_fold = train.iloc[
        train_idx
    ].copy()

    val_fold = train.iloc[
        val_idx
    ].copy()

    # Crear pares únicamente con
    # la parte de entrenamiento
    pares_fold = (
        crear_pares_entrenamiento(
            train_fold
        )
    )

    dataset_fold = (
        preparar_dataset_bert(
            pares_fold,
            entrenamiento=True
        )
    )

    # Se crea un BERT nuevo en cada fold
    modelo_fold = (
        AutoModelForSequenceClassification
        .from_pretrained(
            MODEL_NAME,
            num_labels=2
        )
    )

    argumentos = TrainingArguments(
        output_dir=(
            f"/content/bert_fold_{fold}"
        ),
        num_train_epochs=2,
        per_device_train_batch_size=8,
        learning_rate=2e-5,
        weight_decay=0.01,
        save_strategy="no",
        logging_strategy="epoch",
        report_to="none",
        seed=27
    )

    trainer_fold = Trainer(
        model=modelo_fold,
        args=argumentos,
        train_dataset=dataset_fold,
        data_collator=data_collator
    )

    trainer_fold.train()

    accuracy_fold, _ = evaluar_torneo(
        trainer_fold,
        val_fold
    )

    accuracies_bert_cv.append(
        accuracy_fold
    )

    print(
        f"Fold {fold} - Accuracy@1: "
        f"{accuracy_fold:.2%}"
    )

    del trainer_fold
    del modelo_fold

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


===== FOLD 1 =====


Map:   0%|          | 0/4562 [00:00<?, ? examples/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


... (23 outputs left out)

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
758,0.359686
1516,0.106392


Map:   0%|          | 0/2148 [00:00<?, ? examples/s]

Fold 5 - Accuracy@1: 30.56%


Resumen de los 5 folds:

In [ ]:
print(
    "Accuracy@1 media CV:",
    f"{np.mean(accuracies_bert_cv):.2%}"
)

print(
    "Desviación estándar:",
    f"{np.std(accuracies_bert_cv):.2%}"
)

Accuracy@1 media CV: 34.24%
Desviación estándar: 9.82%


El modelo BERT por pares obtiene una Accuracy@1 media del 34,07 %, con una desviación estándar del 6,09 %. Esto indica que su rendimiento varía entre folds, pero se mantiene en un rango relativamente consistente. En comparación con TF-IDF + Regresión Logística, mejora el rendimiento medio en validación cruzada, lo que sugiere que el enfoque por pares capta mejor las preferencias entre respuestas.

#### Entrenamiento final con todo train

Una vez realizada la validación cruzada, se entrena un nuevo modelo utilizando todas las preguntas disponibles en train.

In [ ]:
pares_train_final = (
    crear_pares_entrenamiento(
        train
    )
)

dataset_train_final = (
    preparar_dataset_bert(
        pares_train_final,
        entrenamiento=True
    )
)

modelo_final = (
    AutoModelForSequenceClassification
    .from_pretrained(
        MODEL_NAME,
        num_labels=2
    )
)

Map:   0%|          | 0/6748 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
argumentos_finales = TrainingArguments(
    output_dir="/content/bert_pairwise_final",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    learning_rate=2e-5,
    weight_decay=0.01,
    save_strategy="no",
    logging_strategy="epoch",
    report_to="none",
    seed=27
)

trainer_final = Trainer(
    model=modelo_final,
    args=argumentos_finales,
    train_dataset=dataset_train_final,
    data_collator=data_collator
)

trainer_final.train()

Step,Training Loss
844,0.331445
1688,0.073252


TrainOutput(global_step=1688, training_loss=0.2023483430039826, metrics={'train_runtime': 1381.6597, 'train_samples_per_second': 9.768, 'train_steps_per_second': 1.222, 'total_flos': 3538695694623360.0, 'train_loss': 0.2023483430039826, 'epoch': 2.0})

#### Evaluación final en test

Finalmente, el modelo se evalúa sobre test, que no ha sido utilizado durante el entrenamiento.

Para cada pregunta se comparan todas sus respuestas dos a dos y se selecciona la que obtiene más victorias.

Accuracy@1 considera la predicción correcta si la respuesta seleccionada tiene target = 1. Por tanto, en las preguntas con empate en el máximo número de upvotes, cualquiera de las respuestas empatadas se considera correcta.

In [ ]:
accuracy_bert_test, ganadores_bert = (
    evaluar_torneo(
        trainer_final,
        test
    )
)

print(
    "BERT por pares - Accuracy@1 test:",
    f"{accuracy_bert_test:.2%}"
)

Map:   0%|          | 0/1666 [00:00<?, ? examples/s]

BERT por pares - Accuracy@1 test: 50.00%


Comparación con los modelos anteriores:

In [ ]:
resultados_finales = pd.DataFrame({
    "Método": [
        "Aleatorio",
        "Respuesta más larga",
        "Respuesta más corta",
        "Solapamiento léxico",
        "TF-IDF + Regresión Logística",
        "BERT por pares"
    ],
    "Accuracy@1 (%)": [
        accuracy_random * 100,
        accuracy_larga * 100,
        accuracy_corta * 100,
        accuracy_solapamiento * 100,
        accuracy_top1 * 100,
        accuracy_bert_test * 100
    ]
})

display(
    resultados_finales.round(2)
)

,Método,Accuracy@1 (%)
0,Aleatorio,39.970000
1,Respuesta más larga,51.790000
2,Respuesta más corta,32.140000
3,Solapamiento léxico,41.070000
4,TF-IDF + Regresión Logística,33.930000
5,BERT por pares,50.000000


El modelo BERT por pares obtiene el mejor resultado en test, con una Accuracy@1 del 53,57 %, superando ligeramente al baseline de respuesta más larga (51,79 %).

El solapamiento léxico alcanza un 41,07 %, el baseline aleatorio un 39,97 %, y TF-IDF + Regresión Logística un 33,93 %.

Estos resultados muestran que el enfoque por pares con BERT es el que mejor rendimiento obtiene entre los métodos evaluados, aunque la mejora respecto al baseline basado en longitud es pequeña.

## BERT por pares ampliado

En el experimento anterior, BERT se entrenó utilizando el mismo conjunto train que los modelos anteriores, del que se habían eliminado completamente las preguntas con empate en el máximo número de upvotes.

Sin embargo, en el enfoque por pares no es necesario eliminar toda la pregunta. Aunque dos respuestas estén empatadas, otras comparaciones de esa misma pregunta pueden seguir aportando información.

Por ejemplo, si:

- A = 5 upvotes
- B = 5 upvotes
- C = 2 upvotes

se descarta únicamente A vs B, pero se pueden utilizar A vs C y B vs C.

Por ello, en este segundo experimento se recuperan las preguntas con empate que pertenecían originalmente a train y se descartan únicamente los pares concretos en los que ambas respuestas tienen el mismo número de upvotes.

El conjunto de test permanece exactamente igual, por lo que los resultados finales de ambos enfoques pueden compararse sobre las mismas preguntas.

#### Recuperación de las preguntas con empate

Aquí usamos train_completo, que era el train antes de eliminar las preguntas con empate top.

In [28]:

print(
    "Preguntas utilizadas en el BERT anterior:",
    train["questionID"].nunique()
)

print(
    "Preguntas disponibles para BERT ampliado:",
    train_completo["questionID"].nunique()
)

print(
    "Preguntas recuperadas:",
    train_completo["questionID"].nunique()
    - train["questionID"].nunique()
)

Preguntas utilizadas en el BERT anterior: 172
Preguntas disponibles para BERT ampliado: 220
Preguntas recuperadas: 48


El nuevo experimento utiliza todas las preguntas asignadas originalmente a train. La función de creación de pares ya elimina automáticamente las comparaciones en las que las dos respuestas tienen exactamente el mismo número de upvotes.

#### Comparación del número de pares

In [29]:
pares_train_anterior = crear_pares_entrenamiento(
    train
)

pares_train_ampliado = crear_pares_entrenamiento(
    train_completo
)

print(
    "Pares en el experimento anterior:",
    len(pares_train_anterior)
)

print(
    "Pares en el experimento ampliado:",
    len(pares_train_ampliado)
)

print(
    "Pares adicionales:",
    len(pares_train_ampliado)
    - len(pares_train_anterior)
)

Pares en el experimento anterior: 6748
Pares en el experimento ampliado: 9356
Pares adicionales: 2608


De esta forma se comprueba que 2608 ejemplos adicionales de entrenamiento se recuperan al conservar las preguntas con empate y eliminar únicamente los pares sin una preferencia clara.

#### Validación cruzada de 5 folds

Aquí creo nuevos folds, porque ahora train_completo contiene más preguntas que el train utilizado anteriormente.

In [35]:
from sklearn.model_selection import GroupKFold

cv_ampliado = GroupKFold(
    n_splits=5
)

cv_splits_ampliado = list(
    cv_ampliado.split(
        train_completo,
        groups=train_completo["questionID"]
    )
)

print(
    "Número de folds:",
    len(cv_splits_ampliado)
)

Número de folds: 5


La validación cruzada se repite utilizando ahora el conjunto de entrenamiento ampliado. En cada fold, las preguntas permanecen agrupadas por questionID.

Para entrenar BERT se utilizan todos los pares con preferencia clara de los cuatro folds de entrenamiento. Los pares empatados se descartan.

El fold restante se utiliza únicamente para validación. En las preguntas con empate en el máximo número de upvotes, cualquiera de las respuestas top se considera correcta al calcular Accuracy@1.

In [ ]:
accuracies_bert_ampliado_cv = []

for fold, (train_idx, val_idx) in enumerate(
    cv_splits_ampliado,
    start=1
):

    print(f"\n===== FOLD {fold} =====")

    train_fold = train_completo.iloc[
        train_idx
    ].copy()

    val_fold = train_completo.iloc[
        val_idx
    ].copy()

    # Se crean pares de entrenamiento.
    # Los pares con los mismos upvotes
    # se descartan automáticamente.
    pares_fold = crear_pares_entrenamiento(
        train_fold
    )

    dataset_fold = preparar_dataset_bert(
        pares_fold,
        entrenamiento=True
    )

    modelo_fold = (
        AutoModelForSequenceClassification
        .from_pretrained(
            MODEL_NAME,
            num_labels=2
        )
    )

    argumentos = TrainingArguments(
        output_dir=(
            f"/content/bert_ampliado_fold_{fold}"
        ),
        num_train_epochs=2,
        per_device_train_batch_size=8,
        learning_rate=2e-5,
        weight_decay=0.01,
        save_strategy="no",
        logging_strategy="epoch",
        report_to="none",
        seed=27
    )

    trainer_fold = Trainer(
        model=modelo_fold,
        args=argumentos,
        train_dataset=dataset_fold,
        data_collator=data_collator
    )

    trainer_fold.train()

    accuracy_fold, _ = evaluar_torneo(
        trainer_fold,
        val_fold
    )

    accuracies_bert_ampliado_cv.append(
        accuracy_fold
    )

    print(
        f"Fold {fold} - Accuracy@1: "
        f"{accuracy_fold:.2%}"
    )

    del trainer_fold
    del modelo_fold

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


===== FOLD 1 =====


Map:   0%|          | 0/6982 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
873,0.409206
1746,0.081600


Map:   0%|          | 0/12462 [00:00<?, ? examples/s]

... (22 outputs left out)

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
984,0.326625
1968,0.091031


Map:   0%|          | 0/4094 [00:00<?, ? examples/s]

Fold 5 - Accuracy@1: 43.48%


In [ ]:
print(
    "Accuracy@1 media CV:",
    f"{np.mean(accuracies_bert_ampliado_cv):.2%}"
)

print(
    "Desviación estándar:",
    f"{np.std(accuracies_bert_ampliado_cv):.2%}"
)

Accuracy@1 media CV: 45.00%
Desviación estándar: 2.76%


El modelo BERT por pares ampliado obtiene una Accuracy@1 media del 45,00 %, con una desviación estándar del 2,76 %. Frente al BERT por pares anterior (34,07 % ± 6,09 %), aumenta el rendimiento medio y reduce la variabilidad entre folds. Esto sugiere que aprovechar las comparaciones válidas de las preguntas con empate aporta información útil para el entrenamiento.

#### Entrenamiento final con el train ampliado

Tras la validación cruzada, se entrena un nuevo modelo BERT utilizando todas las preguntas de train_completo.

Las preguntas con empate se mantienen, pero únicamente se utilizan para el entrenamiento aquellos pares en los que una respuesta tiene más upvotes que la otra.

In [ ]:
pares_train_ampliado = (
    crear_pares_entrenamiento(
        train_completo
    )
)

dataset_train_ampliado = (
    preparar_dataset_bert(
        pares_train_ampliado,
        entrenamiento=True
    )
)

modelo_ampliado = (
    AutoModelForSequenceClassification
    .from_pretrained(
        MODEL_NAME,
        num_labels=2
    )
)

Map:   0%|          | 0/9356 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
argumentos_ampliados = TrainingArguments(
    output_dir="/content/bert_pairwise_ampliado",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    learning_rate=2e-5,
    weight_decay=0.01,
    save_strategy="no",
    logging_strategy="epoch",
    report_to="none",
    seed=27
)

trainer_ampliado = Trainer(
    model=modelo_ampliado,
    args=argumentos_ampliados,
    train_dataset=dataset_train_ampliado,
    data_collator=data_collator
)

trainer_ampliado.train()

#### Evaluación sobre el mismo test

Finalmente, el modelo ampliado se evalúa sobre el mismo conjunto de test utilizado en los experimentos anteriores.

Esto permite comprobar directamente si aprovechar las comparaciones válidas de las preguntas con empate mejora la selección de la respuesta top-1.

In [ ]:
accuracy_bert_ampliado_test, ganadores_bert_ampliado = (
    evaluar_torneo(
        trainer_ampliado,
        test
    )
)

print(
    "BERT por pares ampliado - Accuracy@1 test:",
    f"{accuracy_bert_ampliado_test:.2%}"
)

Map:   0%|          | 0/1666 [00:00<?, ? examples/s]

BERT por pares ampliado - Accuracy@1 test: 57.14%


El BERT por pares ampliado obtiene una Accuracy@1 del 57,14 % en test. Este resultado indica que aprovechar también las comparaciones válidas de las preguntas con empate mejora la capacidad del modelo para seleccionar la respuesta top-1.

#### Comparación de los dos BERT

In [ ]:
comparacion_bert = pd.DataFrame({
    "Modelo": [
        "BERT por pares",
        "BERT por pares ampliado"
    ],
    "Accuracy@1 test (%)": [
        accuracy_bert_test * 100,
        accuracy_bert_ampliado_test * 100
    ]
})

display(
    comparacion_bert.round(2)
)

,Modelo,Accuracy@1 test (%)
0,BERT por pares,53.570000
1,BERT por pares ampliado,57.140000


El BERT por pares ampliado mejora el resultado del modelo anterior, pasando de 53,57 % a 57,14 % de Accuracy@1. Esto supone una mejora de 3,57 puntos porcentuales, por lo que recuperar las preguntas con empate aporta información útil al entrenamiento.

## Análisis de la composición de los folds de validación y test

A partir de la revisión del notebook, se plantea comprobar si los folds de validación y el conjunto de test presentan una composición aproximadamente similar.
En concreto, se analizarán:
- El número de preguntas de cada partición.
- La presencia de preguntas con empate en el máximo número de upvotes.
- el número de respuestas candidatas por pregunta.
El objetivo es comprobar si existen diferencias relevantes entre los folds y test que puedan influir en los resultados de Accuracy@1.
En función de este análisis, se valorará posteriormente si es necesario construir nuevas particiones más equilibradas.

#### Información por pregunta

Primero se construye una tabla con una fila por pregunta, indicando cuántas respuestas candidatas tiene y si existe empate en el máximo número de upvotes.

In [30]:
info_preguntas = (
    df_modelo.groupby("questionID")
    .agg(
        n_respuestas=("answerText", "size")
    )
    .reset_index()
)

n_top = (
    df_modelo.assign(
        es_top=lambda x:
            x["upvotes"]
            == x.groupby("questionID")["upvotes"].transform("max")
    )
    .groupby("questionID")["es_top"]
    .sum()
    .reset_index(name="n_respuestas_top")
)

info_preguntas = info_preguntas.merge(
    n_top,
    on="questionID"
)

info_preguntas["empate"] = (
    info_preguntas["n_respuestas_top"] > 1
)

display(info_preguntas.head())

,questionID,n_respuestas,n_respuestas_top,empate
0,0,86,1,False
1,1,5,1,False
2,2,2,1,False
3,3,3,1,False
4,4,6,1,False


#### Composición de los folds actuales y test

A continuación se compara la composición de los 5 folds de validación con el conjunto de test.

Para cada partición se analiza:
- Número de preguntas.
- Número y porcentaje de preguntas con empate.
- Número medio y mediano de respuestas por pregunta.
- Mínimo y máximo de respuestas por pregunta.

El objetivo es comprobar si existen diferencias relevantes entre las particiones antes de modificar los folds.

In [31]:
def resumen_particion(datos, nombre):

    ids = datos["questionID"].unique()

    info = info_preguntas[
        info_preguntas["questionID"].isin(ids)
    ]

    return {
        "Partición": nombre,
        "Preguntas": len(info),
        "Preguntas con empate": info["empate"].sum(),
        "% con empate": info["empate"].mean() * 100,
        "Media respuestas": info["n_respuestas"].mean(),
        "Mediana respuestas": info["n_respuestas"].median(),
        "Mín. respuestas": info["n_respuestas"].min(),
        "Máx. respuestas": info["n_respuestas"].max()
    }

In [36]:

resumen_particiones = []

for fold, (_, val_idx) in enumerate(
    cv_splits_ampliado,
    start=1
):
    val_fold = train_completo.iloc[val_idx]

    resumen_particiones.append(
        resumen_particion(
            val_fold,
            f"Fold {fold}"
        )
    )

resumen_particiones.append(
    resumen_particion(
        test,
        "Test"
    )
)

resumen_particiones = pd.DataFrame(
    resumen_particiones
)

display(
    resumen_particiones.round(2)
)

,Partición,Preguntas,Preguntas con empate,% con empate,Media respuestas,Mediana respuestas,Mín. respuestas,Máx. respuestas
0,Fold 1,41,6,14.630000,7.020000,3.000000,2,105
1,Fold 2,44,7,15.910000,6.680000,3.000000,2,86
2,Fold 3,40,8,20.000000,6.980000,3.000000,2,82
3,Fold 4,42,13,30.950000,6.790000,3.500000,2,49
4,Fold 5,41,12,29.270000,6.660000,3.000000,2,35
5,Test,54,14,25.930000,4.590000,3.000000,2,18


Los folds tienen un número de preguntas similar entre sí (40–44), por lo que están razonablemente equilibrados en tamaño.
Sin embargo, sí aparecen diferencias en su composición. El porcentaje de preguntas con empate varía desde 14,63 % hasta 30,95 %, mientras que en test es 25,93 %.
También se observan diferencias en el número de respuestas candidatas. Los folds tienen entre 6,66 y 7,02 respuestas de media por pregunta, mientras que test tiene 4,59. Además, algunos folds contienen preguntas con hasta 105 respuestas, frente a un máximo de 18 en test.

#### Distribución del número de respuestas por pregunta
Para observar con mayor detalle las diferencias en el número de respuestas candidatas, las preguntas se agrupan en intervalos según su número de respuestas.

In [37]:
def grupo_n_respuestas(n):
    if n == 2:
        return "2"
    elif n == 3:
        return "3"
    elif n <= 5:
        return "4-5"
    elif n <= 10:
        return "6-10"
    else:
        return ">10"


info_preguntas["grupo_respuestas"] = (
    info_preguntas["n_respuestas"]
    .apply(grupo_n_respuestas)
)

In [38]:
distribuciones = []

for fold, (_, val_idx) in enumerate(
    cv_splits_ampliado,
    start=1
):
    ids_val = (
        train_completo.iloc[val_idx]["questionID"]
        .unique()
    )

    info_val = info_preguntas[
        info_preguntas["questionID"].isin(ids_val)
    ]

    dist = (
        info_val["grupo_respuestas"]
        .value_counts(normalize=True)
        .mul(100)
    )

    dist.name = f"Fold {fold}"
    distribuciones.append(dist)


info_test = info_preguntas[
    info_preguntas["questionID"].isin(
        test["questionID"].unique()
    )
]

dist_test = (
    info_test["grupo_respuestas"]
    .value_counts(normalize=True)
    .mul(100)
)

dist_test.name = "Test"
distribuciones.append(dist_test)

distribucion_respuestas = pd.concat(
    distribuciones,
    axis=1
).fillna(0)

display(
    distribucion_respuestas
    .reindex(["2", "3", "4-5", "6-10", ">10"])
    .round(2)
)

,Fold 1,Fold 2,Fold 3,Fold 4,Fold 5,Test
grupo_respuestas,,,,,,
2,36.590000,36.360000,37.500000,30.950000,34.150000,38.890000
3,19.510000,18.180000,17.500000,19.050000,17.070000,16.670000
4-5,17.070000,15.910000,12.500000,16.670000,14.630000,16.670000
6-10,14.630000,15.910000,20.000000,16.670000,17.070000,22.220000
>10,12.200000,13.640000,12.500000,16.670000,17.070000,5.560000


La distribución confirma que test contiene, en general, preguntas con menos respuestas candidatas. El 38,89 % de las preguntas de test tienen solo 2 respuestas, mientras que en los folds este porcentaje está aproximadamente entre 30,95 % y 37,50 %.
Además, solo el 5,56 % de las preguntas de test tienen más de 10 respuestas, frente a valores entre 12,20 % y 17,07 % en los folds.
Por tanto, test contiene menos preguntas con muchas respuestas candidatas, lo que podría hacer la selección top-1 más sencilla y contribuir a que su Accuracy@1 sea superior a la obtenida en validación cruzada.

#### Combinación de empate y número de respuestas

Antes de reconstruir las particiones, se comprueba cuántas preguntas existen para cada combinación de número de respuestas y presencia de empate. Esto permite verificar si cada grupo contiene suficientes preguntas para poder distribuirlas entre las distintas particiones.

In [40]:
info_preguntas["estrato"] = (
    info_preguntas["grupo_respuestas"]
    + " , empate="
    + info_preguntas["empate"].astype(str)
)

distribucion_estratos = (
    info_preguntas["estrato"]
    .value_counts()
    .sort_index()
)

display(
    distribucion_estratos.to_frame("Número de preguntas")
)

,Número de preguntas
estrato,
"2 , empate=False",78
"2 , empate=True",16
"3 , empate=False",37
"3 , empate=True",10
"4-5 , empate=False",32
"4-5 , empate=True",9
"6-10 , empate=False",34
"6-10 , empate=True",13
">10 , empate=False",21


Todos los estratos contienen suficientes preguntas para intentar realizar una división estratificada. El grupo menos frecuente contiene 9 preguntas, por lo que se pueden mantener aproximadamente las proporciones de número de respuestas y presencia de empates al construir las nuevas particiones.

#### Cambios porpuestos

Para hacer que las particiones sean más comparables, se reconstruirá el split train/test teniendo en cuenta el estrato de cada pregunta, definido por:

- El número de respuestas candidatas (2, 3, 4-5, 6-10, >10).
- La presencia o ausencia de empate en el máximo número de upvotes.

Posteriormente, los 5 folds de validación se construirán utilizando el mismo criterio. Todas las respuestas de una misma pregunta permanecerán siempre juntas.

Una vez creadas las nuevas particiones, se volverá a analizar su composición antes de reentrenar BERT.

Próximos pasos:

- Cambios propuestos

- Nuevo split train-test equilibrado

- Nuevos 5 folds equilibrados

- Comprobación de las nuevas particiones

- Nueva validación cruzada de BERT